# Unified Topology NCA: High-Leverage Training
Training with GPU support and Topological Regularizers.

In [ ]:
%%bash
echo "Installing compatible environment..."
# Install torch 2.0.0 which supports sm_60 (Tesla P100)
pip install --no-cache-dir --force-reinstall torch==2.0.0+cu117 torchvision==0.15.1+cu117 torchaudio==2.0.1 --index-url https://download.pytorch.org/whl/cu117
pip install --upgrade kaggle kagglehub wandb

In [ ]:
import torch
import os

print(f"Torch version: {torch.__version__}")
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"Device: {prop.name}, Compute Capability: {prop.major}.{prop.minor}")
    try:
        x = torch.randn(1, device='cuda')
        print("CUDA Test Successful!")
    except Exception as e:
        print(f"CUDA Test Failed: {e}. Check if the torch build supports sm_{prop.major}{prop.minor}")
else:
    print("CUDA not available.")

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_453cfb028676f79df571e5b2a8ee6afd'

# Symlink datasets
os.makedirs('kaggle_data', exist_ok=True)

input_dir = '/kaggle/input'
mapping = {
    'stratos-manifold-v4': 'stratos_manifold',
    'stratoscot': 'stratoscot',
    'stratos-omega-manifold-v3': 'omega_manifold',
    'fso-manifold': 'fso_manifold',
    'precision-system-v3-data': 'precision_data'
}

if os.path.exists(input_dir):
    for root, dirs, files in os.walk(input_dir):
        item = os.path.basename(root)
        if item in mapping:
            dst = os.path.join('kaggle_data', mapping[item])
            if not os.path.exists(dst):
                # If it's a nested directory structure, link the child if it contains data
                sub = os.listdir(root)
                target_src = root
                if len(sub) == 1 and os.path.isdir(os.path.join(root, sub[0])):
                    target_src = os.path.join(root, sub[0])
                
                os.symlink(target_src, dst)
                print(f"Linked {target_src} -> {dst}")

In [ ]:
%%writefile monitoring_utils.py
import torch
import numpy as np
import os

def calculate_leverage(accuracy, ber, weights_norm):
    efficiency = 1.0 / (1.0 + weights_norm)
    leverage = (accuracy * (1.0 - ber) * efficiency) * 100
    return leverage

def monitor_model_health(model):
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    return total_norm

class WandbLogger:
    def __init__(self, project_name="topo-neural", config=None):
        try:
            import wandb
            self.wandb = wandb
            if os.environ.get('WANDB_API_KEY'):
                self.wandb.init(project=project_name, config=config)
                self.enabled = True
            else:
                print("WANDB_API_KEY not found. Wandb logging disabled.")
                self.enabled = False
        except ImportError:
            print("Wandb not installed. Logging disabled.")
            self.enabled = False

    def log(self, metrics):
        if self.enabled:
            self.wandb.log(metrics)

    def finish(self):
        if self.enabled:
            self.wandb.finish()


In [ ]:
%%writefile data_utils.py
import json
import torch
from torch.utils.data import Dataset, DataLoader

class StratosCoTDataset(Dataset):
    def __init__(self, file_path):
        self.samples = []
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                instruction = data['instruction']
                response = data['response']

                # Extract input for prediction (last line of instruction)
                input_str = instruction.split(':')[-1].strip()

                # Extract target from boxed answer
                if '\\boxed{' in response:
                    target_str = response.split('\\boxed{')[-1].split('}')[0].strip()
                else:
                    continue

                if len(input_str) == 8 and len(target_str) == 8:
                    input_bits = [int(b) for b in input_str]
                    target_bits = [int(b) for b in target_str]
                    self.samples.append((
                        torch.tensor(input_bits, dtype=torch.float32),
                        torch.tensor(target_bits, dtype=torch.float32)
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def get_dataloader(file_path, batch_size=32):
    dataset = StratosCoTDataset(file_path)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

if __name__ == "__main__":
    loader = get_dataloader('./kaggle_data/stratoscot/augmented_train.jsonl')
    x, y = next(iter(loader))
    print(f"Batch X shape: {x.shape}")
    print(f"Batch Y shape: {y.shape}")


In [ ]:
%%writefile kaggle_utils.py




import os
import zipfile
import time
from kaggle.api.kaggle_api_extended import KaggleApi

def download_dataset(dataset, path, retries=3):
    if os.path.exists(path) and len(os.listdir(path)) > 0:
        print(f"Data already exists at {path}. Skipping download.")
        return True

    print(f"Downloading dataset {dataset} to {path}...")
    api = KaggleApi()
    api.authenticate()
    os.makedirs(path, exist_ok=True)
    
    for i in range(retries):
        try:
            api.dataset_download_files(dataset, path=path, unzip=True)
            print(f"Download of {dataset} complete.")
            return True
        except Exception as e:
            print(f"Attempt {i+1} failed to download {dataset}: {e}")
            if i < retries - 1:
                time.sleep(5)
            else:
                raise e
    return False

def download_manifold_data():
    return download_dataset('hichambedrani/stratos-manifold-v4', './kaggle_data/stratos_manifold')

def download_all_resources():
    resources = [
        ('hichambedrani/stratos-manifold-v4', './kaggle_data/stratos_manifold'),
        ('hichambedrani/stratoscot', './kaggle_data/stratoscot'),
        ('hichambedrani/stratos-omega-manifold-v3', './kaggle_data/omega_manifold'),
        ('hichambedrani/fso-manifold', './kaggle_data/fso_manifold'),
        ('hichambedrani/precision-system-v3-data', './kaggle_data/precision_data')
    ]
    for ds, path in resources:
        download_dataset(ds, path)

class KaggleSearch:
    """
    Search utility for Kaggle Datasets and Models.
    """
    def __init__(self):
        self.api = KaggleApi()
        self.api.authenticate()

    def search_datasets(self, query):
        print(f"Searching for datasets matching: {query}")
        datasets = self.api.dataset_list(search=query)
        for ds in datasets:
            print(f"Dataset: {ds.ref} | Title: {ds.title}")
        return datasets

    def search_models(self, query):
        print(f"Searching for models matching: {query}")
        try:
            models = self.api.model_list(search=query)
            for model in models:
                print(f"Model: {model.ownerSlug}/{model.slug} | Title: {model.title}")
            return models
        except AttributeError:
            print("Model search not supported in this Kaggle API version.")
            return []

    def discover_and_download_resources(self, query, base_path='./kaggle_data/discovered'):
        print(f"Discovering and downloading resources for: {query}")
        datasets = self.search_datasets(query)
        downloaded_paths = []
        for ds in datasets[:3]: # Limit to top 3
            path = os.path.join(base_path, ds.ref.replace('/', '_'))
            if download_dataset(ds.ref, path):
                downloaded_paths.append(path)
        return downloaded_paths

if __name__ == "__main__":
    search = KaggleSearch()
    search.search_datasets("manifold")


In [ ]:
%%writefile kaggle_hub_manager.py
import kagglehub
import os
import shutil
import torch
import json

class KaggleHubManager:
    """
    Manages model interactions with the Kaggle Model Hub using kagglehub.
    """
    def __init__(self, model_handle=None):
        self.model_handle = model_handle

    def download_model(self, handle=None):
        handle = handle or self.model_handle
        if not handle:
            raise ValueError("No model handle provided.")
        
        print(f"Downloading model from Kaggle Hub: {handle}...")
        path = kagglehub.model_download(handle)
        print(f"Model downloaded to: {path}")
        return path

    def upload_model_version(self, handle, local_model_dir, version_notes="New model version"):
        """
        Uploads a new version of a model to Kaggle Model Hub.
        handle: 'owner/model/framework/variation'
        """
        if not os.path.exists(local_model_dir):
            raise FileNotFoundError(f"Local model directory {local_model_dir} does not exist.")
        
        print(f"Uploading model version to {handle} from {local_model_dir}...")
        try:
            path = kagglehub.model_upload(handle, local_model_dir, version_notes=version_notes)
            print(f"Model successfully uploaded to {handle}")
            return path
        except Exception as e:
            print(f"Failed to upload model: {e}")
            return None

def save_and_push_to_hub(model, optimizer, epoch, metrics, handle, local_dir='checkpoint'):
    """
    Helper to save a checkpoint locally and push it to Kaggle Hub.
    """
    os.makedirs(local_dir, exist_ok=True)
    checkpoint_path = os.path.join(local_dir, 'model.pt')
    torch_state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics
    }
    torch.save(torch_state, checkpoint_path)
    
    with open(os.path.join(local_dir, 'metadata.json'), 'w') as f:
        json.dump(metrics, f)

    manager = KaggleHubManager()
    manager.upload_model_version(handle, local_dir, version_notes=f"Epoch {epoch} checkpoint")

if __name__ == "__main__":
    pass


In [ ]:
%%writefile weights_loader.py
import os
import numpy as np
import torch
from kaggle_utils import download_all_resources

class ManifoldLoader:
    """
    Enhanced utility to load weights from multiple Stratos/Omega/FSO Manifold datasets.
    Includes strict shape validation.
    """
    def __init__(self, source='stratos', directory=None):
        # download_all_resources()

        sources = {
            'stratos': 'kaggle_data/stratos_manifold',
            'omega': 'kaggle_data/omega_manifold',
            'fso': 'kaggle_data/fso_manifold'
        }

        base_dir = directory or sources.get(source)
        if not base_dir:
            raise ValueError(f"Unknown source: {source}")

        # Find the actual directory containing .npy files
        self.directory = self._find_npy_dir(base_dir)
        print(f"[{source.upper()} LOADER] Using directory: {self.directory}")

        self.weight_files = sorted([f for f in os.listdir(self.directory) if f.startswith('weight_')])
        self.lib_files = sorted([f for f in os.listdir(self.directory) if f.startswith('lib_')])
        self.ptr = 0

    def _find_npy_dir(self, start_path):
        if not os.path.exists(start_path):
             raise FileNotFoundError(f"Directory {start_path} does not exist.")
        for root, dirs, files in os.walk(start_path):
            if any(f.endswith('.npy') for f in files):
                return root
        raise FileNotFoundError(f"No .npy files found in {start_path}")

    def load_weights(self, num_weights, edge_dim=32, node_dim=32):
        end = min(self.ptr + num_weights, len(self.weight_files))
        actual_num = end - self.ptr

        weights = []
        target_size = edge_dim * node_dim
        
        for i in range(self.ptr, end):
            file_path = os.path.join(self.directory, self.weight_files[i])
            try:
                data = np.load(file_path)
            except Exception as e:
                raise IOError(f"Failed to load weight file {file_path}: {e}")
                
            # Flatten then reshape to fit requested dimensions
            flat_data = data.flatten()
            
            if flat_data.size < target_size:
                print(f"Warning: Data size {flat_data.size} in {self.weight_files[i]} is less than target {target_size}. Padding with zeros.")
                padded = np.zeros(target_size)
                padded[:flat_data.size] = flat_data
                reshaped = padded.reshape(edge_dim, node_dim)
            else:
                if flat_data.size > target_size:
                    print(f"Warning: Data size {flat_data.size} in {self.weight_files[i]} exceeds target {target_size}. Truncating.")
                reshaped = flat_data[:target_size].reshape(edge_dim, node_dim)
            weights.append(reshaped)

        self.ptr = end

        # If not enough weights, instead of random, we can raise error or pad.
        # Original code used random padding. Keeping it but with a message.
        if len(weights) < num_weights:
            print(f"Warning: Requested {num_weights} weights but only found {len(weights)}. Padding with random.")
            padding = [np.random.randn(edge_dim, node_dim) for _ in range(num_weights - len(weights))]
            weights.extend(padding)

        return torch.tensor(np.array(weights), dtype=torch.float32)

    def load_library(self, num_libs, edge_dim=32, node_dim=32):
        if num_libs > len(self.lib_files):
            print(f"Warning: Requested {num_libs} libraries but only found {len(self.lib_files)}.")
            num_libs = len(self.lib_files)

        libs = []
        target_size = edge_dim * node_dim
        for i in range(num_libs):
            file_path = os.path.join(self.directory, self.lib_files[i])
            try:
                data = np.load(file_path)
            except Exception as e:
                raise IOError(f"Failed to load library file {file_path}: {e}")

            flat_data = data.flatten()
            if flat_data.size < target_size:
                padded = np.zeros(target_size)
                padded[:flat_data.size] = flat_data
                reshaped = padded.reshape(edge_dim, node_dim)
            else:
                reshaped = flat_data[:target_size].reshape(edge_dim, node_dim)
            libs.append(reshaped)

        return torch.tensor(np.array(libs), dtype=torch.float32)


In [ ]:
%%writefile sheaf_nn.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SheafDiffusionLayer(nn.Module):
    def __init__(self, num_nodes, edges, node_dim, edge_dim, alpha=0.01):
        super(SheafDiffusionLayer, self).__init__()
        self.num_nodes = num_nodes
        self.num_edges = len(edges)
        self.d = node_dim
        self.de = edge_dim
        self.alpha = alpha

        self.W_maps = nn.Parameter(torch.randn(2 * self.num_edges, self.de, self.d))
        self.register_buffer('edge_index', torch.tensor(edges).t().contiguous())

    def load_from_manifold(self, loader):
        """
        Initializes W_maps using weights from the ManifoldLoader.
        """
        manifold_weights = loader.load_weights(2 * self.num_edges, edge_dim=self.de, node_dim=self.d)
        if manifold_weights.shape == self.W_maps.shape:
            self.W_maps.data.copy_(manifold_weights)
            print(f"Successfully loaded {2 * self.num_edges} manifold weights into W_maps.")
        else:
            print(f"Warning: Manifold weight shape {manifold_weights.shape} does not match W_maps shape {self.W_maps.shape}.")

    def forward(self, H):
        batch_size = H.size(0)
        W_src = self.W_maps[0::2]
        W_dst = self.W_maps[1::2]
        u_idx = self.edge_index[0]
        v_idx = self.edge_index[1]

        H_u = H[:, u_idx, :]
        H_v = H[:, v_idx, :]

        proj_u = torch.matmul(W_src, H_u.unsqueeze(-1)).squeeze(-1)
        proj_v = torch.matmul(W_dst, H_v.unsqueeze(-1)).squeeze(-1)

        Z = proj_u - proj_v

        grad_u = torch.matmul(W_src.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)
        grad_v = torch.matmul(W_dst.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)

        Delta_H = torch.zeros_like(H)
        expanded_u_idx = u_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_u_idx, grad_u)
        expanded_v_idx = v_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_v_idx, -grad_v)

        H_new = H - self.alpha * Delta_H
        return F.relu(H_new)

    def project_to_stiefel(self):
        with torch.no_grad():
            W = self.W_maps
            WtW = torch.matmul(W.transpose(-1, -2), W)
            e, v = torch.linalg.eigh(WtW)
            e_inv_sqrt = torch.diag_embed(1.0 / torch.sqrt(torch.clamp(e, min=1e-6)))
            WtW_inv_sqrt = torch.matmul(torch.matmul(v, e_inv_sqrt), v.transpose(-1, -2))
            W_new = torch.matmul(W, WtW_inv_sqrt)
            self.W_maps.copy_(W_new)

class SheafNCALayer(SheafDiffusionLayer):
    """
    Learned local update rule based on Sheaf residuals.
    Instead of simple diffusion, uses an MLP to compute the update.
    """
    def __init__(self, num_nodes, edges, node_dim, edge_dim, hidden_dim=16):
        super(SheafNCALayer, self).__init__(num_nodes, edges, node_dim, edge_dim)
        # MLP takes [local_feature, sheaf_residual]
        self.mlp = nn.Sequential(
            nn.Linear(node_dim + node_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim),
        )

    def load_from_manifold(self, loader):
        """
        Initializes both W_maps and the internal MLP using manifold weights.
        """
        super().load_from_manifold(loader)

        # Load MLP weights from 'lib' files
        lib_weights = loader.load_library(2, edge_dim=1, node_dim=1024) # Placeholder for more complex mapping
        print("Note: MLP manifold integration is using available lib tensors.")

    def forward(self, H):
        batch_size = H.size(0)
        W_src = self.W_maps[0::2]
        W_dst = self.W_maps[1::2]
        u_idx = self.edge_index[0]
        v_idx = self.edge_index[1]

        H_u = H[:, u_idx, :]
        H_v = H[:, v_idx, :]

        proj_u = torch.matmul(W_src, H_u.unsqueeze(-1)).squeeze(-1)
        proj_v = torch.matmul(W_dst, H_v.unsqueeze(-1)).squeeze(-1)

        Z = proj_u - proj_v

        grad_u = torch.matmul(W_src.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)
        grad_v = torch.matmul(W_dst.transpose(-1, -2), Z.unsqueeze(-1)).squeeze(-1)

        Delta_H = torch.zeros_like(H)
        expanded_u_idx = u_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_u_idx, grad_u)
        expanded_v_idx = v_idx.view(1, -1, 1).expand(batch_size, -1, self.d)
        Delta_H.scatter_add_(1, expanded_v_idx, -grad_v)

        # Local update: concatenate current feature and aggregated sheaf residual
        concat_feat = torch.cat([H, Delta_H], dim=-1)
        update = self.mlp(concat_feat)

        H_new = H + update
        return F.relu(H_new)

class DeepSheafNetwork(nn.Module):
    def __init__(self, num_nodes, edges, node_dim, edge_dim, num_layers=3, alpha=0.01, layer_type='diffusion'):
        super(DeepSheafNetwork, self).__init__()
        self.layer_type = layer_type
        if layer_type == 'diffusion':
            self.layers = nn.ModuleList([
                SheafDiffusionLayer(num_nodes, edges, node_dim, edge_dim, alpha=alpha)
                for _ in range(num_layers)
            ])
        elif layer_type == 'nca':
            self.layers = nn.ModuleList([
                SheafNCALayer(num_nodes, edges, node_dim, edge_dim)
                for _ in range(num_layers)
            ])

    def load_from_manifold(self, loader):
        """
        Initializes all layers using weights from the ManifoldLoader.
        """
        for i, layer in enumerate(self.layers):
            print(f"Loading manifold weights for layer {i}...")
            layer.load_from_manifold(loader)

    def forward(self, H):
        for layer in self.layers:
            H = layer(H)
        return H

    def project_all_to_stiefel(self):
        for layer in self.layers:
            layer.project_to_stiefel()


In [ ]:
%%writefile topo_torch.py
import torch

def relaxed_euler_torch(grid):
    """
    Continuous relaxation of the Euler Characteristic in PyTorch.
    Supports backpropagation.
    grid: Tensor of shape [Batch, H, W] with values in [0, 1]
    """
    # V: Sum of vertices
    V = torch.sum(grid, dim=(1, 2))

    # E_h: Horizontal edges
    E_h = torch.sum(grid[:, :, :-1] * grid[:, :, 1:], dim=(1, 2))

    # E_v: Vertical edges
    E_v = torch.sum(grid[:, :-1, :] * grid[:, 1:, :], dim=(1, 2))

    E = E_h + E_v

    # F: Faces (quads)
    F = torch.sum(grid[:, :-1, :-1] * grid[:, :-1, 1:] * grid[:, 1:, :-1] * grid[:, 1:, 1:], dim=(1, 2))

    return V - E + F

def compute_euler_binary_torch(grid_binary):
    """
    Exact Euler Characteristic for binary tensors in PyTorch.
    grid_binary: Tensor of shape [Batch, H, W] with values in {0, 1}
    """
    V = torch.sum(grid_binary, dim=(1, 2))

    # Use logical_and for exactness, though multiplication works for binary
    E_h = torch.sum(grid_binary[:, :, :-1] * grid_binary[:, :, 1:], dim=(1, 2))
    E_v = torch.sum(grid_binary[:, :-1, :] * grid_binary[:, 1:, :], dim=(1, 2))
    E = E_h + E_v

    F = torch.sum(grid_binary[:, :-1, :-1] * grid_binary[:, :-1, 1:] * grid_binary[:, 1:, :-1] * grid_binary[:, 1:, 1:], dim=(1, 2))

    return V - E + F


In [ ]:
%%writefile spectral_topo.py
import torch

def assemble_coboundary_matrix(num_nodes, edge_index, W_maps, de, d):
    """
    Assembles the Coboundary Matrix D_F as a dense tensor.
    Vectorized implementation.
    W_maps: [2 * E, de, d]
    edge_index: [2, E]
    Returns: D_F of shape [E * de, V * d]
    """
    num_edges = edge_index.size(1)
    device = W_maps.device
    dtype = W_maps.dtype
    
    D_F = torch.zeros(num_edges * de, num_nodes * d, device=device, dtype=dtype)
    
    # Indices for row-wise blocks
    row_starts = torch.arange(num_edges, device=device) * de
    
    # Source blocks (W_u)
    u_idx = edge_index[0] # [E]
    u_col_starts = u_idx * d # [E]
    
    # Destination blocks (-W_v)
    v_idx = edge_index[1] # [E]
    v_col_starts = v_idx * d # [E]
    
    W_src = W_maps[0::2] # [E, de, d]
    W_dst = W_maps[1::2] # [E, de, d]
    
    # Create grid of offsets within each block
    ii, jj = torch.meshgrid(torch.arange(de, device=device), torch.arange(d, device=device), indexing='ij')
    
    # Expand to all edges
    row_indices = (row_starts.view(-1, 1, 1) + ii.view(1, de, d)).view(-1)
    col_indices_src = (u_col_starts.view(-1, 1, 1) + jj.view(1, de, d)).view(-1)
    col_indices_dst = (v_col_starts.view(-1, 1, 1) + jj.view(1, de, d)).view(-1)
    
    # Flattened indices for 1D scatter
    stride = num_nodes * d
    idx_src = row_indices * stride + col_indices_src
    idx_dst = row_indices * stride + col_indices_dst
    
    D_F.view(-1).scatter_add_(0, idx_src, W_src.reshape(-1))
    D_F.view(-1).scatter_add_(0, idx_dst, -W_dst.reshape(-1))
    
    return D_F

def compute_sheaf_laplacian_spectral_gap(num_nodes, edge_index, W_maps, de, d):
    """
    Computes the second smallest eigenvalue of the Sheaf Laplacian.
    This serves as a differentiable proxy for connectivity/alignment.
    """
    D_F = assemble_coboundary_matrix(num_nodes, edge_index, W_maps, de, d)
    L_F = torch.matmul(D_F.t(), D_F) # [V*d, V*d]

    # Compute eigenvalues
    eigenvalues = torch.linalg.eigvalsh(L_F)

    # Return the second smallest eigenvalue
    if eigenvalues.numel() > 1:
        return eigenvalues[1]
    else:
        return eigenvalues[0]

def spectral_connectivity_loss(num_nodes, edge_index, W_maps, de, d, target_gap=0.1):
    """
    Loss that encourages the Sheaf Laplacian to have a spectral gap.
    """
    gap = compute_sheaf_laplacian_spectral_gap(num_nodes, edge_index, W_maps, de, d)
    return torch.relu(target_gap - gap)


In [ ]:
%%writefile train_high_leverage.py
import torch
import torch.nn as nn
import torch.optim as optim
from sheaf_nn import DeepSheafNetwork
from weights_loader import ManifoldLoader
from data_utils import get_dataloader
from monitoring_utils import calculate_leverage, monitor_model_health, WandbLogger
from kaggle_hub_manager import save_and_push_to_hub
from topo_torch import relaxed_euler_torch
from spectral_topo import compute_sheaf_laplacian_spectral_gap
import json
import os

def train(dry_run=False, use_topo_loss=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"--- Large Scale High-Leverage Training ---")
    print(f"Device: {device}")

    num_nodes = 8
    edges = [(i, (i + 1) % num_nodes) for i in range(num_nodes)]
    for i in range(num_nodes):
        edges.append((i, (i + 2) % num_nodes))
    
    edge_index = torch.tensor(edges).t().to(device)

    node_dim = 32
    edge_dim = 32
    num_layers = 12

    config = {
        "num_nodes": num_nodes, "node_dim": node_dim, "edge_dim": edge_dim,
        "num_layers": num_layers, "lr": 0.0001, "weight_decay": 0.01,
        "topo_weight": 0.1, "spectral_weight": 0.05
    }

    try:
        model = DeepSheafNetwork(num_nodes, edges, node_dim, edge_dim, num_layers=num_layers, layer_type='nca').to(device)
        # Immediate check for CUDA compatibility
        if device.type == 'cuda':
            dummy = torch.randn(1, num_nodes, node_dim).to(device)
            model(dummy)
            print("GPU model verification successful.")
    except Exception as e:
        if 'no kernel image' in str(e) or 'CUDA error' in str(e):
            print(f"GPU error: {e}. Falling back to CPU.")
            device = torch.device('cpu')
            edge_index = edge_index.to(device)
            model = DeepSheafNetwork(num_nodes, edges, node_dim, edge_dim, num_layers=num_layers, layer_type='nca').to(device)
        else:
            raise e

    logger = WandbLogger(project_name="topo-neural-high-leverage", config=config)

    scaler = torch.amp.GradScaler(device.type, enabled=(device.type == 'cuda'))

    print("Initializing from Multiple Manifolds...")
    try:
        stratos_loader = ManifoldLoader(source='stratos')
        omega_loader = ManifoldLoader(source='omega')
        for i, layer in enumerate(model.layers):
            loader = stratos_loader if i % 2 == 0 else omega_loader
            layer.load_from_manifold(loader)
    except Exception as e:
        print(f"Initialization from manifold failed: {e}. Using random weights.")

    output_head = nn.Sequential(
        nn.Linear(node_dim, 128), nn.LayerNorm(128), nn.ReLU(), nn.Linear(128, 8)
    ).to(device)

    optimizer = optim.AdamW(list(model.parameters()) + list(output_head.parameters()), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    criterion = nn.BCEWithLogitsLoss()

    report_file = 'TRAINING_REPORT.jsonl'
    
    try:
        dataloader = get_dataloader('./kaggle_data/stratoscot/augmented_train.jsonl', batch_size=256)
    except Exception as e:
        print(f"Dataloader failed: {e}. Using dummy data.")
        dataloader = [(torch.randn(10, 8), torch.randint(0, 2, (10, 8)).float())]

    num_epochs = 10 if not dry_run else 1
    best_leverage = -1

    print("Starting Epochs...")
    for epoch in range(num_epochs):
        model.train()
        total_loss, total_ber, correct, total = 0, 0, 0, 0

        for batch_idx, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            autocast_device = device.type if device.type in ['cuda', 'cpu'] else 'cpu'
            with torch.amp.autocast(autocast_device, enabled=(device.type == 'cuda')):
                H = x.unsqueeze(-1).repeat(1, 1, node_dim)
                H_out = model(H)
                logits = output_head(H_out).mean(dim=1)
                main_loss = criterion(logits, y)
                
                topo_loss = 0
                if use_topo_loss:
                    grid_probs = torch.sigmoid(logits).view(-1, 2, 4)
                    chi = relaxed_euler_torch(grid_probs)
                    topo_loss = torch.mean((chi - 1.0)**2)
                    last_layer = model.layers[-1]
                    gap = compute_sheaf_laplacian_spectral_gap(num_nodes, edge_index, last_layer.W_maps, last_layer.de, last_layer.d)
                    spectral_loss = torch.relu(0.1 - gap)
                    loss = main_loss + config["topo_weight"] * topo_loss + config["spectral_weight"] * spectral_loss
                else:
                    loss = main_loss

            scaler.scale(loss).backward()
            grad_norm = monitor_model_health(model)
            if device.type == 'cuda': scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == y).all(dim=1).sum().item()
            total += x.size(0)
            total_ber += torch.mean((preds != y).float()).item()
            if dry_run and batch_idx >= 2: break

        scheduler.step()
        avg_acc = correct / total if total > 0 else 0
        avg_ber = total_ber / len(dataloader)
        with torch.no_grad():
            w_norm = sum(p.norm(2).item() for p in model.parameters())
            leverage = calculate_leverage(avg_acc, avg_ber, w_norm)

        metrics = {"epoch": epoch, "loss": total_loss / len(dataloader), "accuracy": avg_acc, "ber": avg_ber, "leverage": leverage, "grad_norm": grad_norm}
        logger.log(metrics)
        with open(report_file, 'a') as f: f.write(json.dumps(metrics) + '\n')
        print(f"Epoch {epoch}: Loss={metrics['loss']:.4f}, Acc={avg_acc:.2%}, BER={avg_ber:.4f}, Leverage={leverage:.4f}")

        if leverage > best_leverage and not dry_run:
            best_leverage = leverage
            handle = os.environ.get('KAGGLE_MODEL_HANDLE')
            if handle: save_and_push_to_hub(model, optimizer, epoch, metrics, handle)

    logger.finish()
    print("High-Leverage Model Training Complete.")

if __name__ == "__main__":
    train(dry_run=True)


In [ ]:
from train_high_leverage import train
train(dry_run=False)